# VneuroTK 路径系统
VneuroTK 用三个路径类统一管理不同数据源的文件定位：

 | 类 | 用途 |
 |---|---|
 | `EphysPath` | Ephys 数据（spike raster / 平均发放率等） |
 | `MNEPath` | MNE-BIDS 格式的 MEG / EEG 数据 |
 | `VTKPath` | VneuroTK 自有 HDF5 保存格式 |

 路径类只负责**构造文件路径**，不触发任何 IO 操作。
 读取数据统一通过 `vtk.read(path_obj)` 完成。

**本示例覆盖**：
1. `EphysPath` — 基本用法与支持的数据类型
2. `EphysPath` — 多 session、多 probe 命名规则
3. `EphysPath.from_components` — 拆解 session_id 的构造方式
4. `EphysPath` — 辅助路径属性
5. `MNEPath` — MEG/EEG BIDS 路径
6. `VTKPath` — HDF5 保存路径

In [ ]:
from pathlib import Path

from vneurotk.datasets import sample
from vneurotk.io import EphysPath, MNEPath, VTKPath

_PROJECT_ROOT = Path.cwd().parents[1]
EPHYS_ROOT = _PROJECT_ROOT / "DB" / "ephys" / "MonkeyVision"
MNE_ROOT = Path("/nfs/t2/datasets/NOD/MEEG/NOD-MEEG/NOD-MEG") / "derivatives" / "preprocessed" / "raw"
SAVE_ROOT = _PROJECT_ROOT / "DB" / "saved"

EPHYS_ROOT = sample.data_path()

  0%|                                              | 0.00/87.2M [00:00<?, ?B/s]

## 1. EphysPath — 基本用法

`EphysPath(root, session_id, dtype, extension)` 构造单条数据文件路径。

### 支持的数据类型（`dtype`）

| dtype | 说明 | 级别 |
|---|---|---|
| `TrialRaster` | 完整 spike raster（时序）| unit |
| `TrialRecord` | trial 行为/元数据表 | unit |
| `UnitProp` | 单元属性（位置、质量等）| unit |
| `MeanFr` | 各单元平均发放率 | unit |
| `ChTrialRaster` | channel-level trial raster | channel |
| `ChTrialRecord` | channel-level trial 元数据 | channel |
| `ChMeanFr` | channel-level 平均发放率 | channel |
| `ChProp` | channel 属性 | channel |
| `ChStimFr` | channel-level 刺激期间发放率 | channel |

In [ ]:
SES1 = "251024_FanFan_nsd1w_MSB"

# --- unit-level ---
for dtype, ext in [
    ("TrialRaster", "h5"),
    ("TrialRecord", "csv"),
    ("UnitProp", "csv"),
    ("MeanFr", "h5"),
]:
    p = EphysPath(root=EPHYS_ROOT, session_id=SES1, dtype=dtype, extension=ext)
    print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/TrialRecord_251024_FanFan_nsd1w_MSB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/UnitProp_251024_FanFan_nsd1w_MSB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/MeanFr_251024_FanFan_nsd1w_MSB.h5


In [ ]:
# --- channel-level ---
for dtype, ext in [
    ("ChTrialRaster", "h5"),
    ("ChTrialRecord", "csv"),
    ("ChMeanFr", "h5"),
    ("ChProp", "csv"),
    ("ChStimFr", "h5"),
]:
    p = EphysPath(root=EPHYS_ROOT, session_id=SES1, dtype=dtype, extension=ext)
    print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/ChTrialRaster_251024_FanFan_nsd1w_MSB.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/ChTrialRecord_251024_FanFan_nsd1w_MSB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/ChMeanFr_251024_FanFan_nsd1w_MSB.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/ChProp_251024_FanFan_nsd1w_MSB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/ChStimFr_251024_FanFan_nsd1w_MSB.h5


## 2. 多 session 与多 probe

### Session ID 格式
```
{date}_{subject}_{paradigm}_{region}
```
`region` 可含连字符（如 `AL-ASB`），路径构造不受影响。

In [ ]:
SES3 = "251112_ZhuangZhuang_nsd1w_AL-ASB"

for dtype, ext in [
    ("TrialRaster", "h5"),
    ("TrialRecord", "csv"),
    ("UnitProp", "csv"),
    ("MeanFr", "h5"),
    ("ChTrialRecord", "csv"),
]:
    p = EphysPath(root=EPHYS_ROOT, session_id=SES3, dtype=dtype, extension=ext)
    print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/TrialRaster_251112_ZhuangZhuang_nsd1w_AL-ASB.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/TrialRecord_251112_ZhuangZhuang_nsd1w_AL-ASB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/UnitProp_251112_ZhuangZhuang_nsd1w_AL-ASB.csv
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/MeanFr_251112_ZhuangZhuang_nsd1w_AL-ASB.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/ChTrialRecord_251112_ZhuangZhuang_nsd1w_AL-ASB.csv


### 多 probe 命名规则

同一 session 使用多根电极时，文件名末尾追加 `_probe{N}`（N 从 0 开始）：
```
{dtype}_{session_id}_probe{N}.{ext}
```

In [ ]:
for probe in (0, 1):
    p = EphysPath(root=EPHYS_ROOT, session_id=SES1, dtype="TrialRaster", probe=probe, extension="h5")
    print(p.fpath)

for probe in (0, 1):
    p = EphysPath(root=EPHYS_ROOT, session_id=SES1, dtype="MeanFr", probe=probe, extension="h5")
    print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB_probe0.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB_probe1.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/MeanFr_251024_FanFan_nsd1w_MSB_probe0.h5
/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/MeanFr_251024_FanFan_nsd1w_MSB_probe1.h5


## 3. `from_components` 构造方法

当 session_id 可分解为标准四字段时，推荐使用 `from_components`——
可读性更好，且能在构造时验证字段格式。

In [ ]:
p = EphysPath.from_components(
    root=EPHYS_ROOT,
    date="251024",
    subject="FanFan",
    paradigm="nsd1w",
    region="MSB",
    dtype="TrialRaster",
    extension="h5",
)
print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB.h5


In [ ]:
p = EphysPath.from_components(
    root=EPHYS_ROOT,
    date="251112",
    subject="ZhuangZhuang",
    paradigm="nsd1w",
    region="AL-ASB",
    dtype="MeanFr",
    extension="h5",
)
print(p.fpath)

/nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251112_ZhuangZhuang_nsd1w_AL-ASB/MeanFr_251112_ZhuangZhuang_nsd1w_AL-ASB.h5


## 4. 辅助路径属性

| 属性 | 路径 | 说明 |
|---|---|---|
| `session_dir` | `{root}/sessions/{session_id}` | session 数据目录 |
| `raw_dir` | `{root}/raw/{session_id}` | 原始数据目录 |
| `nwb_path` | `{root}/nwb/{session_id}[_probe{N}].nwb` | NWB 文件路径 |

In [ ]:
p = EphysPath(root=EPHYS_ROOT, session_id=SES1, dtype="TrialRaster")
print("session_dir :", p.session_dir)
print("raw_dir     :", p.raw_dir)
print("nwb_path    :", p.nwb_path)  # 单电极，无 probe 后缀

# 多电极 nwb 路径
p_probe = EphysPath(root=EPHYS_ROOT, session_id=SES1, probe=0)
print("nwb probe0  :", p_probe.nwb_path)

session_dir : /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/sessions/251024_FanFan_nsd1w_MSB
raw_dir     : /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/raw/251024_FanFan_nsd1w_MSB
nwb_path    : /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/nwb/251024_FanFan_nsd1w_MSB.nwb
nwb probe0  : /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/ephys/MonkeyVision/nwb/251024_FanFan_nsd1w_MSB_probe0.nwb


## 5. MNEPath — MEG/EEG BIDS 路径

遵循 MNE-BIDS 目录约定：
```
{root}/sub-{subject}/ses-{session}/meg/
    sub-{subject}_ses-{session}_task-{task}_run-{run}_{suffix}.{extension}
```

In [ ]:
mne_path = MNEPath(
    root=MNE_ROOT,
    subject="01",
    session="ImageNet01",
    task="ImageNet",
    run="01",
    suffix="meg_clean",
    extension=".fif",
)
print("MEG 文件:", mne_path.fpath)

MEG 文件: /nfs/t2/datasets/NOD/MEEG/NOD-MEEG/NOD-MEG/derivatives/preprocessed/raw/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif


## 6. VTKPath — HDF5 保存路径

VneuroTK 自有的保存格式，以 BIDS-like 命名规则构造 `.h5` 文件路径：
```
{root}/sub-{subject}_ses-{session}_task-{task}_run-{run}.h5
```

In [ ]:
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

vtk_path = VTKPath(
    SAVE_ROOT,
    subject="01",
    session="ImageNet01",
    task="ImageNet",
    run="01",
)
print("VTK 文件:", vtk_path.fpath)

VTK 文件: /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/saved/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5


`VTKPath` 也可以直接从现有 `.h5` 文件路径构造：

In [ ]:
existing_h5 = SAVE_ROOT / "sub-01_ses-ImageNet01_task-ImageNet_run-01.h5"
vtk_from_path = VTKPath(existing_h5)
print("从路径构造:", vtk_from_path.fpath)

从路径构造: /nfs/t1/userhome/zzl-zgh/workingdir/repos/vntk/DB/saved/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5/data.h5
